
# Citi Bike Data Ingestion (Multi-Month)
Downloads, unzips, normalizes, and saves Citi Bike trip data for a configurable
date range. Handles the schema change that occurred in February 2021.
**Storage estimate (Community Edition — 15GB limit):**
| Range | ~Size | Recommendation |
|---|---|---|
| 1 month | 2–4 GB | ✅ Safe |
| 2 months | 4–8 GB | ✅ Safe |
| 3 months | 6–12 GB | ⚠️ Watch space |
| 4+ months | 10+ GB | ❌ Risk of hitting limit |


## 0. Config — set date range

In [0]:

import pyspark.sql.functions as psf
from pyspark.sql.functions import col, count, avg, max, min, round as spark_round
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import requests, zipfile, gc
from io import BytesIO
from datetime import date
from dateutil.relativedelta import relativedelta
import pandas as pd

In [0]:
START_YEAR  = 2025        # inclusive
START_MONTH = 7
END_YEAR    = 2025        # inclusive
END_MONTH   = 9

DB_NAME    = "citibike"
TABLE_NAME = "trips"
FULL_TABLE = f"{DB_NAME}.{TABLE_NAME}"

In [0]:
def month_range(sy, sm, ey, em):
    cur = date(sy, sm, 1)
    end = date(ey, em, 1)
    while cur <= end:
        yield cur.year, cur.month
        cur += relativedelta(months=1)

months = list(month_range(START_YEAR, START_MONTH, END_YEAR, END_MONTH))
print(f"Months to process: {len(months)}  |  {months[0]} → {months[-1]}")

## 1. Download & extract

In [0]:
URL_PATTERNS = [
    "https://s3.amazonaws.com/tripdata/{y}{m:02d}-citibike-tripdata.csv.zip",
    "https://s3.amazonaws.com/tripdata/{y}{m:02d}-citibike-tripdata.zip",
]

def fetch_zip_bytes(year, month):
    """Download zip from S3 and return raw bytes, or None if all patterns fail."""
    for pattern in URL_PATTERNS:
        url = pattern.format(y=year, m=month)
        try:
            r = requests.get(url, timeout=180)
            if r.status_code == 200:
                print(f"  ✅ {year}-{month:02d} fetched ({len(r.content)/1e6:.1f} MB)")
                return r.content
            print(f"  ⚠️  HTTP {r.status_code}: {url.split('/')[-1]}")
        except Exception as e:
            print(f"  ⚠️  {url.split('/')[-1]}: {e}")
    print(f"  ❌ {year}-{month:02d} — all patterns failed")
    return None

def zip_bytes_to_pandas(zip_bytes):
    """
    Extract all CSVs from zip bytes entirely in memory.
    Returns a single concatenated pandas DataFrame.
    """
    dfs = []
    with zipfile.ZipFile(BytesIO(zip_bytes)) as zf:
        csv_names = [n for n in zf.namelist() if n.endswith(".csv")]
        for name in csv_names:
            with zf.open(name) as f:
                dfs.append(pd.read_csv(f, low_memory=False))
    return pd.concat(dfs, ignore_index=True) if dfs else None


## 2. Load & normalize schema

Citi Bike changed column names in **Feb 2021**:

| Pre-2021 (old) | Post-2021 (new) |
|---|---|
| `tripduration` | computed from timestamps |
| `starttime` | `started_at` |
| `stoptime` | `ended_at` |
| `start station id` | `start_station_id` |
| `start station name` | `start_station_name` |
| `start station latitude` | `start_lat` |
| `usertype` (Subscriber/Customer) | `member_casual` (member/casual) |

We normalise everything to the **new schema** below.

In [0]:

KEEP_COLS = [
    "ride_id", "rideable_type",
    "started_at", "ended_at",
    "start_station_name", "start_station_id",
    "end_station_name",   "end_station_id",
    "start_lat", "start_lng",
    "end_lat",   "end_lng",
    "member_casual",
]

def is_old_schema(pdf):
    return "starttime" in pdf.columns or "start station name" in pdf.columns

def normalise_pandas(pdf):
    """Rename old-schema columns to new unified names in pandas."""
    if not is_old_schema(pdf):
        # New schema — just keep the cols we care about
        available = [c for c in KEEP_COLS if c in pdf.columns]
        return pdf[available].copy()

    pdf = pdf.rename(columns={
        "bikeid":                   "ride_id",
        "starttime":                "started_at",
        "stoptime":                 "ended_at",
        "start station name":       "start_station_name",
        "start station id":         "start_station_id",
        "end station name":         "end_station_name",
        "end station id":           "end_station_id",
        "start station latitude":   "start_lat",
        "start station longitude":  "start_lng",
        "end station latitude":     "end_lat",
        "end station longitude":    "end_lng",
    })
    pdf["rideable_type"] = "classic_bike"
    pdf["member_casual"] = pdf["usertype"].map(
        {"Subscriber": "member", "Customer": "casual"}
    ).fillna("casual")

    available = [c for c in KEEP_COLS if c in pdf.columns]
    return pdf[available].copy()

## 3. Build unified Delta table

In [0]:
# Create the database once (safe to re-run)
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
print(f"Database '{DB_NAME}' ready.")

In [0]:
written = 0
for y, m in months:
    print(f"\nProcessing {y}-{m:02d}...")

    # Step 1 — download to memory
    zip_bytes = fetch_zip_bytes(y, m)
    if zip_bytes is None:
        continue

    # Step 2 — extract + normalise entirely in pandas (no filesystem)
    pdf = zip_bytes_to_pandas(zip_bytes)
    del zip_bytes    # free memory immediately
    gc.collect()

    if pdf is None or pdf.empty:
        print(f"  ⚠️  No CSV data found in zip for {y}-{m:02d}")
        continue

    pdf = normalise_pandas(pdf)

    # Step 3 — cast types in pandas before handing to Spark
    pdf["started_at"] = pd.to_datetime(pdf["started_at"], errors="coerce")
    pdf["ended_at"]   = pd.to_datetime(pdf["ended_at"],   errors="coerce")
    pdf["start_lat"]  = pd.to_numeric(pdf["start_lat"],   errors="coerce")
    pdf["start_lng"]  = pd.to_numeric(pdf["start_lng"],   errors="coerce")
    pdf["end_lat"]    = pd.to_numeric(pdf["end_lat"],     errors="coerce")
    pdf["end_lng"]    = pd.to_numeric(pdf["end_lng"],     errors="coerce")

    # Drop nulls on key fields
    pdf = pdf.dropna(subset=["started_at", "start_station_name"])
    print(f"  Rows after cleaning: {len(pdf):,}")

    # Step 4 — pandas → Spark → saveAsTable (no path, no filesystem)
    sdf = spark.createDataFrame(pdf)
    del pdf
    gc.collect()

    mode = "overwrite" if written == 0 else "append"
    sdf.write.format("delta").mode(mode).saveAsTable(FULL_TABLE)
    print(f"  ✅ Saved to {FULL_TABLE}  (mode={mode})")
    written += 1
    del sdf
    gc.collect()

print(f"\n{'='*50}")
print(f"Done. {written}/{len(months)} months written to '{FULL_TABLE}'")

## 4. Register temp view + row count

In [0]:
# Any subsequent notebook loads the table with just:
#   df = spark.table("citibike.trips")
#   df.createOrReplaceTempView("trips")

df = spark.table(FULL_TABLE)
# df.createOrReplaceTempView("trips")
# print(f"Total rows: {df.count():,}")

In [0]:
print(f"Total rows: {df.count():,}")

## 5. Basic EDA

In [0]:
# Rides per year-month
df.groupBy(
    psf.year("started_at").alias("year"),
    psf.month("started_at").alias("month")
).agg(count("*").alias("total_rides")) \
 .orderBy("year", "month") \
 .show(20)

In [0]:
# Top 10 start stations
df.groupBy("start_station_name") \
  .count() \
  .orderBy("count", ascending=False) \
  .limit(10) \
  .show(truncate=False)

In [0]:
# Top 10 end stations
df.groupBy("end_station_name") \
  .count() \
  .orderBy("count", ascending=False) \
  .limit(10) \
  .show(truncate=False)


In [0]:
# Ride duration stats by member type
df.withColumn(
    "duration_min",
    (col("ended_at").cast("long") - col("started_at").cast("long")) / 60
).filter((col("duration_min") > 0) & (col("duration_min") < 1440)) \
 .groupBy("member_casual") \
 .agg(
     spark_round(avg("duration_min"), 2).alias("avg_min"),
     spark_round(psf.percentile_approx("duration_min", 0.5), 2).alias("median_min"),
     spark_round(max("duration_min"), 2).alias("max_min"),
     count("*").alias("n_rides")
 ).show()


In [0]:
# Ride type breakdown (classic vs electric)
df.groupBy("rideable_type") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()


In [0]:
# ----------------------------------------------------------------